In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab10.ipynb")

<img src="data6.png" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Lab 10 – TF-IDF, Cohen's Kappa, Macro F1

## Data 6, Summer 2025

In this lab, you'll explore a dataset of online requests for free pizza. Your goals:

- Calculate TF-IDF scores 
- Evaluate inter-rater agreement using Cohen’s Kappa
- Implement Macro F1

This lab is due on **Sunday, at 11:00 PM**. You must submit the assignment to Gradescope.

In [ ]:
from datascience import *
import numpy as np
import pandas as pd
from tqdm import tqdm

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Random Acts of Pizza

In this lab, we will be looking at data collected from the reddit community [Random Acts of Pizza](https://www.reddit.com/r/Random_Acts_Of_Pizza/). Every row in our data, `pizza`, is a request to this community for pizza. For the purpose of this lab, we have chosen three columns:

* `request_id`: identifier of the post on Reddit, e.g. "t3_w5491"
* `request_text`: full text of the reddit post
* `requester_received_pizza`: boolean indicating if the user received pizza or not

In [ ]:
pizza = Table.read_table("pizza_acts.csv")
pizza = pizza.select("request_id", "request_text", "requester_received_pizza")
pizza

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Section 1: TF-IDF

To better understand the Random Acts of Pizza requests, we’ll use **TF-IDF** to find which words stand out in different kinds of posts.

<hr style="border: 1px solid #fdb515;" />

## Question 1a – Word Frequency

Let's start by making a dictionary of word frequencies. Using `pizza`, create a dictionary `word_counts` that maps all the unique words in all of the text to their corresponding frequencies. For instance, across the 1000 rows, if the word "thanks" shows up 50 times, we should have an entry in our dictionary with the key "thanks" and the corresponding value 50. Additionaly, we will consider "Thanks" and "thanks" as the same word.

**_Hint:_** Look into the string `split` method [here](https://data6.org/su25/reference/#string-methods) and string `lower` method [here](https://www.w3schools.com/python/ref_string_lower.asp).

In [ ]:
word_counts = {} 

for request in pizza.column("request_text"):
    words = ...  # lowercase + split on whitespace
    for word in words:
        if ... not in ...:
            word_counts[word] = ...
        ...

# This sorts our dictionary based off the corresponding frequency, and turns it into a list.
word_counts = sorted(word_counts.items(), key=lambda item: item[1], reverse = True)
word_counts[:20]  # Display the top 20 most frequent words

In [ ]:
grader.check("q1a")

<hr style="border: 1px solid #fdb515;" />

## Question 1b - Reflection

Suppose we try using `word_counts` to get a sense of what people are talking about. In 1-2 sentences, what is a potential downside of this method?

_Type your answer here, replacing this text._

<hr style="border: 1px solid #fdb515;" />

## TF-IDF

To try to get more semantically significant terms (words that carry more meaning), let's compute a TF-IDF score for each word within `request_text`. TF-IDF stands for term frequency-inverse document frequency. In other words, we will count the frequency of some term (term frequency), and multiply it by the inverse of the number of times that same term appears across the entire dataset. Intuitively, this will give higher scores to words that appear multiple times in a given reddit post, but also don't appear that often across your entire dataset.

You'll start by computing the term frequency, which follows this formula:

$$
\text{TF}(w, r) = \frac{\text{count}(w, r)}{\text{total\_words}(r)}
$$

**Description**:
- `w`: The word being analyzed.
- `r`: The specific Reddit post you're examining.
- `count(w, r)`: The number of occurrences of `w` in Reddit post `r`.
- `total_words(r)`: The total number of words in the Reddit post `r`.

---

Then, you'll calculate the inverse document frequency, following this formula:

$$
\text{IDF}(w) = \log \left( \frac{N}{1 + \text{df}(w, lst)} \right)
$$

**Description**:
- `w`: The word being analyzed.
- `N`: The total number of Reddit posts in the dataset (in this case, 4040).
- `df(w, lst)`: The number of Reddit posts containing the word `w` in `lst`.

<hr style="border: 1px solid #fdb515;" />

## Question 1c – `count`

**_Note:_** Do not use the built-in Python function `count`.

In [ ]:
def count(w, r):
    total = ...
    ...



count("hello", "hello hello hello\n ") # Should return 3 

In [ ]:
grader.check("q1c")

<hr style="border: 1px solid #fdb515;" />

## Question 1d – `total_words`

In [ ]:
def total_words(r):
    """
    Returns the number of words in the Post r, separated by whitespace.

    >>> post = "hello there friend!"
    >>> total_words(post)
    3
    """
    ...

total_words("hello there friend!")  # Should return 3

In [ ]:
grader.check("q1d")

<hr style="border: 1px solid #fdb515;" />

## Question 1e – `df`

In [ ]:
example = "I only want the first hi - hi hi hi"

In [ ]:
for word in example.split():
    if word == "hi":
        print(word)

In [ ]:
for word in example.split(): 
    if word == "hi":
        print(word)
        break # Notice "hi" is only printed once now!

To understand what `df` should return, it's the number of Reddit posts containing the word `w` in the entire dataset `d` when looking at column `c`.

For example, the word "hello" appears in 3 reddit post across d.column(c), return 3.

Example: 

    Post 1: "hello hello"
    Post 2: "Nope."
    Post 3: "Nah."
    Post 4: "hello there!"
    Post 5: "hello"
    Post 6: "hellooo"

We expect 3 to be returned (Post 1, 4, and 5).

In [ ]:
def df(w, lst):
    count = ...
    for doc in lst:
        if ... in ...:
            ... 
    ...

docs = ["hello hello",  "Nope.", "Nah.", "hello there!", "hello", "hellooo"]
df("hello", docs) # Should return 3 because it counts how many documents contain the word "hello"

In [ ]:
grader.check("q1e")

<hr style="border: 1px solid #fdb515;" />

## Question 1f – `tf_idf`

Finally, fill in the function to compute the TF-IDF. As a reminder, here are all the formulas:

$$
\text{TF}(w, r) = \frac{\text{count}(w, r)}{\text{total\_words}(r)}
$$

$$
\text{IDF}(w, d, c) = \log \left( \frac{N}{1 + \text{df}(w, lst)} \right)
$$

$$
\text{TF-IDF}(w, r, lst) = \text{TF}(w, r) \times \text{IDF}(w, lst)
$$

In [ ]:
# We us np.log to get the natural log of some number and avoid division by zero.
def tf_idf(w, r, lst):
    """
    Compute the tf_idf score for word w within reddit post r.
    To compute the inverse document frequency, use lst, the set of documents.
    """
    # Term Frequency (TF)
    tf = ...

    # Inverse Document Frequency (IDF)
    N = ...
    df_w = ...
    idf = np.log(N /(1 + df_w))
    ...

tf_idf("hello", "hello there friend!", docs)  # Should return a TF-IDF score for the word "hello"

In [ ]:
grader.check("q1f")

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

## Question 1g – Understanding TF-IDF Formula

**Why do we add 1 to the denominator of the IDF formula?**  What could go wrong if we didn’t include the `+1` when computing inverse document frequency?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 1px solid #fdb515;" />

## Demo – `word_highest_tf_idf`

In this section, we use the `tf_idf` function to compare each word in the entire Reddit pizza dataset to two reference documents, the Bee Movie script and the U.S. Constitution. We treat the entire Reddit dataset as one large document and compute the TF-IDF score for each unique word using a 3-document corpus (`[pizza_text, bee_text, constitution_text]`).

We then build a dictionary called `word_highest_tf_idf`, where the keys are words, and the values represent how many times that word had the **highest TF-IDF score** seen so far during the iteration. However, because we are computing TF-IDF scores on the full dataset rather than individual Reddit posts, this count simply reflects which words scored highest **globally**, not per post.

For example, if the TF-IDF scores are:
```python

{"Hi": 1, "there": 0.5, "friend": 1}
```

and

```python
{"Hi": 0.3, "friend": 0.1}
```

we should expect the following dictionary assigned to `word_highest_tf_idf`:

```python
{"Hi": 2, "friend": 1}
```

This is because "Hi" had the highest TF-IDF score in a reddit post twice, and friend had the highest TF-IDF score in a reddit post once.

**_Note_**: We're using the `tqdm` library which gives us a progress bar, showing how long this process will take. This chunk of code took around 5 minutes to run for staff.

---

Now for the following, we do the above. We have converted the code into markdown because this took ~30 minutes to run. Focus on the output of `word_highest_tf_idf`.

---

```python 
# Convert txt files into a string
with open("pizza.txt", "r") as file:
    pizza_text = file.read()

with open("beemovie.txt", "r") as file:
    bee_text = file.read()

with open("constitution.txt", "r") as file:
    constitution_text = file.read()

pizza_bee_constitution = [pizza_text, bee_text, constitution_text]
bee_text

word_highest_tf_idf = {}
pizza_requests = column_to_string(pizza.column("request_text"))
word_scores = {}

# Iterate through each unique word in the pizza dataset
for word in tqdm(set(pizza_requests.split()), desc="Processing TF-IDF scores: "):
    score = tf_idf(word, pizza_requests, pizza_bee_constitution) # TF-IDF for that word, in the entire Reddit dataset, using the 3-document corpus
    word_scores[word] = score # Store the TF-IDF for that word
    
    max_score = max(word_scores.values()) # Calculate the maximum score
    
    for word, score in word_scores.items(): # Find the single highest TF-IDF score seen so far
        if score == max_score:
            if word not in word_highest_tf_idf:
                word_highest_tf_idf[word] = 1
            else:
                word_highest_tf_idf[word] += 1

word_highest_tf_idf = sorted(word_highest_tf_idf.items(), key=lambda item: item[1], reverse=True)
word_highest_tf_idf

[('i', 13460),
 ('pizza', 11084),
 ('college', 374),
 ('site', 84),
 ('strong', 35),
 ('Love', 34),
 ('"trade"', 6),
 ('me!!', 6),
 ('apperciate', 5),
 ('delectable', 4),
 ('Surrey,BC', 3),
 ('otis:', 2),
 ('met', 1),
 ('scamming', 1)]
 ```

<hr style="border: 1px solid #fdb515;" />

## Question 1h – Reflection

How does the TF-IDF value compare to straight up word count in terms of usefulness? Any patterns you're noticing? 3-4 sentences is sufficient for this section.

_Type your answer here, replacing this text._

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Section 2 – Cohen's Kappa and Codebook

**Cohen’s Kappa** is a measure of agreement between two annotators (or raters) who independently classify items into categories.

Unlike simple agreement rates (e.g., how often the two raters agree), **Cohen’s Kappa adjusts for chance agreement,** that is, how often two people might agree just by random guessing.


## Cohen’s Kappa Formula

$[
\kappa = \frac{P_o - P_e}{1 - P_e}
]$

Where:
- $(P_o)$ = observed agreement (how often the raters agreed)
- $(P_e)$ = expected agreement by chance (based on the marginals)

##  2x2 Confusion Matrix Format

Imagine two annotators label 10 Reddit posts as either `"Yes"` or `"No"` depending on whether the post appeals to **emotion**. Their annotations can be summarized like this:

![2x2.png](2x2.png)

This means:
- Both said **Yes** on 3 posts.
- One said **Yes** and their partner said **No** on 1 post.
- One said **No** and their partner said **Yes** on 1 posts.
- Both said **No** on 5 posts.

<hr style="border: 1px solid #fdb515;" />

## Question 2a – Cohen's Kappa

Now, let's implement Cohen's Kappa using the formula above. 

In [ ]:
def cohen_kappa(a_and_b_yes, a_yes_b_no, a_no_b_yes, a_and_b_no):
    """
    Computes Cohen's Kappa from a 2x2 confusion matrix.
    """
    total = ...
    po = ...
    # Compute marginal probabilities

    a_yes = ...
    a_no = ...
    b_yes = ...
    b_no = ...

    pe = ...

    kappa = ...
    ...

# Example: 3 agree on Yes, 5 agree on No, rest disagree
cohen_kappa(3, 1, 1, 4)

In [ ]:
grader.check("q2a")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Section 3 – Macro F1 Score

## Question 3a – Implementing Macro F1

Now that we’ve imported the precision and recall functions from the `sklearn` library, your task is to implement the F1 score for a single class, and then compute the **macro F1**, which is the unweighted average of F1 scores across all classes. Even though you don't have to implement precision and recall, here are all the formulas:

---

### Precision
$$
\text{Precision} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}} = \frac{TP}{TP + FP}
$$

---

### Recall
$$
\text{Recall} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}} = \frac{TP}{TP + FN}
$$

---

### F1 Score
$$
\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

---

### Macro F1 Score
$$
\text{Macro F1} = \frac{1}{N} \sum_{i=1}^{N} \text{F1}_i
$$

Where:
- \( TP \) = True Positives  
- \( FP \) = False Positives  
- \( FN \) = False Negatives  
- \( N \) = Number of classes

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Sample ground truth and predicted labels
y_true = ["Yes", "Yes", "Yes", "No", "No", "No", "No", "Yes", "No", "No"]
y_pred = ["Yes", "No",  "Yes", "No", "Yes", "No", "No", "Yes", "No", "No"]

# Labels for consistency in output order
labels = ["Yes", "No"]

# Compute per-class precision, recall, and F1 using sklearn
precisions = precision_score(y_true, y_pred, labels=labels, average=None)
recalls = recall_score(y_true, y_pred, labels=labels, average=None)
print(precisions, recalls) 

In [ ]:
# Compute F1 score for each class
def f1(prec, rec):
    if prec + rec == 0:
        return 0
    return ...

def macro_f1(precisions, recalls):
    f1_scores = ...
    for i in ...:
        ...
    return ...
# Run it
macro_f1(precisions, recalls) # Should be about 0.792

In [ ]:
grader.check("3a")

## Pets of Data 6

Portobello makes a comeback! The last lab :)

<img src="portobello_caramello.png" width="50%" alt="Cute pets"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)